# Notebook 02 — Lógica Fuzzy aplicada à base de diabetes

Neste notebook, vamos usar a lógica fuzzy para representar **graus de pertencimento**.

Na lógica clássica, uma afirmação costuma ser verdadeira ou falsa.  
Na lógica fuzzy, uma afirmação pode ser verdadeira em certo grau.

Exemplo didático:

- risco baixo: grau próximo de 0;
- risco intermediário: grau próximo de 0,5;
- risco alto: grau próximo de 1.

Para a base de diabetes, isso permite transformar sintomas em um escore gradual de suspeita.

## Etapa 1 — Enviar o arquivo da base

In [ ]:
# ============================================================
# 1. Faça upload do arquivo diabetes_data_upload.csv
# ============================================================

from google.colab import files
import io
import pandas as pd

print("Selecione o arquivo diabetes_data_upload.csv no seu computador.")
uploaded = files.upload()

# Pega automaticamente o primeiro arquivo enviado
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print("Arquivo carregado com sucesso!")
print(f"Nome do arquivo: {nome_arquivo}")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

## Etapa 2 — Preparar funções auxiliares

A lógica fuzzy precisa de valores numéricos para calcular graus.

Por isso, vamos converter respostas `Yes` e `No` em 1 e 0, além de criar uma função simples de pertinência para a idade.

In [ ]:
# ============================================================
# Bibliotecas e funções auxiliares
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Coluna-alvo da base
TARGET = "class"

# Marcadores clínico-sintomáticos presentes na base
SINTOMAS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]

# Sintomas escolhidos como centrais para simular lacunas ou conflito
# A escolha é didática: são sintomas que aparecem com relevância clínica
# e ajudam a mostrar o comportamento das lógicas.
SINTOMAS_CENTRAIS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "partial paresis",
]

# Pesos heurísticos.
# Eles não foram aprendidos por machine learning.
# São apenas uma forma didática de dizer:
# "alguns sintomas pesam mais na suspeita do que outros".
PESOS = {
    "Polyuria": 3.0,
    "Polydipsia": 3.0,
    "sudden weight loss": 2.0,
    "weakness": 1.0,
    "Polyphagia": 1.5,
    "Genital thrush": 0.5,
    "visual blurring": 1.0,
    "Itching": 0.25,
    "Irritability": 1.0,
    "delayed healing": 0.75,
    "partial paresis": 2.0,
    "muscle stiffness": 0.5,
    "Alopecia": 0.25,
    "Obesity": 0.25,
}

def sim_nao_para_numero(valor):
    '''
    Converte respostas categóricas da base para números.

    Yes -> 1
    No  -> 0
    NaN -> NaN

    Essa conversão permite que a lógica opere sobre evidências.
    '''
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()

    if valor == "yes":
        return 1.0
    elif valor == "no":
        return 0.0
    else:
        return np.nan

def pertinencia_idade(idade):
    '''
    Transforma idade em uma evidência gradual simples.

    Menos de 30 anos: pertinência 0
    Entre 30 e 60 anos: crescimento linear
    Acima de 60 anos: pertinência 1

    Não é uma regra médica definitiva.
    É apenas uma simplificação didática para mostrar como
    variáveis numéricas podem entrar no raciocínio lógico.
    '''
    if pd.isna(idade):
        return 0.5

    if idade < 30:
        return 0.0
    elif idade > 60:
        return 1.0
    else:
        return (idade - 30) / 30

def classe_para_binario(valor):
    '''
    Converte a classe original da base para comparação simples.

    Positive -> 1
    Negative -> 0
    '''
    return 1 if valor == "Positive" else 0

def mostrar_distribuicao_decisoes(resultado, coluna="decisao", titulo="Distribuição das decisões"):
    '''
    Plota um gráfico simples com a distribuição das saídas produzidas pela lógica.
    '''
    contagem = resultado[coluna].value_counts()

    plt.figure(figsize=(8, 4))
    barras = plt.bar(contagem.index.astype(str), contagem.values)
    plt.title(titulo)
    plt.xlabel("Saída produzida pela lógica")
    plt.ylabel("Quantidade de registros")
    plt.xticks(rotation=25, ha="right")

    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            str(int(altura)),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

def resumo_simples(resultado, coluna_decisao="decisao"):
    '''
    Gera um resumo didático da saída da lógica.

    A ideia não é tratar a lógica como um classificador tradicional,
    mas observar como ela decide, quando se abstém e como distribui
    suas respostas.
    '''
    total = len(resultado)

    resumo = resultado[coluna_decisao].value_counts().reset_index()
    resumo.columns = ["decisao", "quantidade"]
    resumo["percentual"] = 100 * resumo["quantidade"] / total

    return resumo

## Etapa 3 — Conhecer a base

In [ ]:
# ============================================================
# Olhando a base antes de aplicar qualquer lógica
# ============================================================

print("Dimensão da base:", df.shape)
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nDistribuição da classe:")
display(df[TARGET].value_counts().reset_index().rename(columns={"index": "classe", TARGET: "quantidade"}))

print("\nQuantidade de valores ausentes por coluna:")
display(df.isna().sum().reset_index().rename(columns={"index": "coluna", 0: "ausentes"}))

print("\nResumo da idade:")
display(df["Age"].describe())

## Etapa 4 — Ideia da lógica fuzzy

A lógica fuzzy calcula um **grau de risco** entre 0 e 1.

Neste notebook:

- sintomas presentes aumentam o grau de suspeita;
- sintomas ausentes reduzem o grau de suspeita;
- sintomas ausentes por falta de informação recebem valor neutro `0,5`;
- a idade entra como uma pertinência gradual.

A saída será:

- `baixa_suspeita`;
- `suspeita_moderada`;
- `alta_suspeita`.

In [ ]:
def decisao_fuzzy(linha):
    soma_ponderada = 0
    soma_pesos = 0
    quantidade_lacunas = 0

    for sintoma in SINTOMAS:
        valor = sim_nao_para_numero(linha[sintoma])

        # Na lógica fuzzy, uma informação ausente pode ser tratada
        # como pertinência neutra, isto é, nem totalmente sim nem totalmente não.
        if pd.isna(valor):
            valor = 0.5
            quantidade_lacunas += 1

        peso = PESOS[sintoma]
        soma_ponderada += valor * peso
        soma_pesos += peso

    # Idade como componente gradual
    soma_ponderada += pertinencia_idade(linha["Age"]) * 0.75
    soma_pesos += 0.75

    grau_risco = soma_ponderada / soma_pesos

    if grau_risco >= 0.65:
        decisao = "alta_suspeita"
    elif grau_risco >= 0.35:
        decisao = "suspeita_moderada"
    else:
        decisao = "baixa_suspeita"

    return pd.Series({
        "grau_risco_fuzzy": grau_risco,
        "quantidade_lacunas": quantidade_lacunas,
        "decisao": decisao
    })

def aplicar_logica_fuzzy(base):
    resultado = base.copy()
    saidas = resultado.apply(decisao_fuzzy, axis=1)
    resultado = pd.concat([resultado, saidas], axis=1)
    resultado["classe_binaria"] = resultado[TARGET].apply(lambda x: 1 if x == "Positive" else 0)
    return resultado

## Etapa 5 — Aplicar a lógica fuzzy na base original

Observe que a principal saída é o `grau_risco_fuzzy`.

Ele não é apenas uma classe final, mas um grau contínuo de suspeita.

In [ ]:
resultado_original = aplicar_logica_fuzzy(df)

display(resultado_original[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "sudden weight loss", "weakness",
    "class", "grau_risco_fuzzy",
    "quantidade_lacunas", "decisao"
]].head(10))

display(resumo_simples(resultado_original))
mostrar_distribuicao_decisoes(
    resultado_original,
    titulo="Lógica fuzzy — base original"
)

## Etapa 6 — Simular dados incompletos

Agora vamos apagar parte dos sintomas centrais.

A lógica fuzzy tende a continuar produzindo uma resposta, pois consegue lidar com valores intermediários.

In [ ]:
# ============================================================
# Criando uma cópia da base com lacunas simuladas
# ============================================================

def criar_base_incompleta(base, taxa_lacunas=0.20, semente=42):
    '''
    Cria uma versão da base com alguns sintomas centrais apagados.

    Isso simula uma situação comum em saúde:
    nem toda informação foi registrada, perguntada ou confirmada.
    '''
    base_incompleta = base.copy()
    rng = np.random.default_rng(semente)

    for coluna in SINTOMAS_CENTRAIS:
        mascara = rng.random(len(base_incompleta)) < taxa_lacunas
        base_incompleta.loc[mascara, coluna] = np.nan

    return base_incompleta

df_incompleta = criar_base_incompleta(df, taxa_lacunas=0.20)

print("Base original:")
display(df[SINTOMAS_CENTRAIS].head())

print("Base com lacunas simuladas:")
display(df_incompleta[SINTOMAS_CENTRAIS].head())

print("Valores ausentes após simulação:")
display(df_incompleta[SINTOMAS_CENTRAIS].isna().sum())

In [ ]:
resultado_incompleto = aplicar_logica_fuzzy(df_incompleta)

display(resultado_incompleto[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "sudden weight loss", "weakness",
    "class", "grau_risco_fuzzy",
    "quantidade_lacunas", "decisao"
]].head(10))

display(resumo_simples(resultado_incompleto))
mostrar_distribuicao_decisoes(
    resultado_incompleto,
    titulo="Lógica fuzzy — base com lacunas"
)

## Etapa 7 — Visualizar o grau de risco

Além da decisão textual, podemos olhar a distribuição do grau fuzzy.

Isso mostra a vantagem da lógica fuzzy: ela preserva uma saída gradual.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(resultado_original["grau_risco_fuzzy"], bins=20)
plt.title("Distribuição do grau de risco fuzzy")
plt.xlabel("Grau de risco fuzzy")
plt.ylabel("Quantidade de registros")
plt.tight_layout()
plt.show()

## Etapa 8 — Reflexão

A lógica fuzzy é útil quando o problema não tem fronteiras rígidas.

Em vez de dizer apenas `risco` ou `não risco`, ela permite dizer:

> Este registro pertence ao conjunto de alta suspeita com certo grau.

Isso combina bem com situações clínicas em que a evidência aparece de forma gradual.

In [ ]:
resultado_original.to_csv("resultado_fuzzy_original.csv", index=False)
resultado_incompleto.to_csv("resultado_fuzzy_incompleta.csv", index=False)

print("Arquivos exportados:")
print("- resultado_fuzzy_original.csv")
print("- resultado_fuzzy_incompleta.csv")